Quark/Gluon Jet Dataset
========================
Loads the ParticleNet quark/gluon jet dataset (Zenodo record 3164691) and
converts each jet from a zero-padded particle array into a graph-structured
PyTorch Geometric Data object suitable for GNN-based classification.

Graph construction rationale
------------------------------
A jet is a collimated spray of particles produced by a hard-scattered quark
or gluon. Representing it as a set of independent, unconnected particles
ignores the fact that particles produced nearby in (η, φ) space share a
common origin — either from the same QCD splitting or from soft wide-angle
radiation. Angular proximity in (Δη, Δφ) therefore serves as a physically
motivated prior for inter-particle correlations.

Construction steps applied to each jet:
  1. Remove zero-padded entries (particles with pT < 1e-6 GeV).
  2. Convert raw Cartesian 4-momenta (px, py, pz, E) to cylindrical
     coordinates (pT, η, φ, E) and compute jet-centred relative quantities
     (Δη, Δφ) using the pT-weighted centroid.
  3. Build a directed k = 16 nearest-neighbour graph in (Δη, Δφ) space.
     The choice k = 16 matches the original ParticleNet hyperparameter and
     provides enough neighbourhood context for multi-scale feature learning
     while avoiding over-smoothing in deeper GNN layers.
  4. Assemble a six-dimensional node feature vector per particle:
         [Δη, Δφ, log(pT), log(E), log(pT / ΣpT), log(E / ΣE)]
     Logarithmic scaling of energy quantities compresses the dynamic range
     (which can span several orders of magnitude) and empirically improves
     both gradient flow and final model accuracy.

Expected HDF5 file schema
--------------------------
  - X : float32  (N_jets, N_particles, 4)   —  [px, py, pz, E] per particle
  - y : int32    (N_jets,)                  —  0 → gluon,  1 → quark

Download:
  https://zenodo.org/records/3164691

In [1]:
!pip install h5py
!pip install torch
!pip install torch_geometric
!pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.6 MB/s eta 0:00:00


In [9]:
import numpy as np
import h5py
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from sklearn.neighbors import NearestNeighbors

# ---------------------------------------------------------------------------
# Coordinate utilities
# ---------------------------------------------------------------------------

def _to_cylindrical(px, py, pz, E):
    """Convert Cartesian 4-momenta to (pT, pseudorapidity η, azimuth φ, E)."""
    pT = np.sqrt(px ** 2 + py ** 2)
    p_mag = np.sqrt(px ** 2 + py ** 2 + pz ** 2) + 1e-9
    eta = np.arctanh(np.clip(pz / p_mag, -1 + 1e-7, 1 - 1e-7))
    phi = np.arctan2(py, px)
    return pT, eta, phi, E

def _looks_like_particlenet_pt_eta_phi_pid(raw: np.ndarray) -> bool:
    """
    Heuristic detector for the common ParticleNet tensor schema:
        [pT, η, φ, pid]
    The 4th channel (`pid`) is typically integer-valued with magnitudes around
    standard PDG IDs (e.g. 22, 130, 211, 2212).  This makes it easy to
    distinguish from Cartesian energy, which is continuous.
    """
    ch3 = raw[:, 3]
    finite = np.isfinite(ch3)
    if finite.sum() == 0:
        return False
    vals = ch3[finite]
    frac_part = np.abs(vals - np.round(vals))
    near_integer_ratio = np.mean(frac_part < 1e-6)
    return near_integer_ratio > 0.90 and np.max(np.abs(vals)) > 100

def _extract_kinematics(raw: np.ndarray):
    """
    Return (pT, η, φ, E) from either supported schema:
    1) Cartesian 4-momenta: [px, py, pz, E]
    2) ParticleNet features: [pT, η, φ, pid]
    For schema (2), E is approximated using the massless relation
    E ≈ pT * cosh(η), which is sufficient for stable feature construction.
    """
    if _looks_like_particlenet_pt_eta_phi_pid(raw):
        pT = np.clip(raw[:, 0], 0.0, None)
        eta = raw[:, 1]
        phi = _wrap_phi(raw[:, 2])
        E = pT * np.cosh(np.clip(eta, -8, 8))
        return pT, eta, phi, E
    px, py, pz, E = raw[:, 0], raw[:, 1], raw[:, 2], raw[:, 3]
    return _to_cylindrical(px, py, pz, E)

def _wrap_phi(dphi):
    """Wrap an angular difference to the interval (−π, π]."""
    return (dphi + np.pi) % (2 * np.pi) - np.pi

def _node_features(particles: np.ndarray) -> np.ndarray:
    """
    Compute six-dimensional node features from raw Cartesian 4-momenta.
    The jet centroid is defined as the pT-weighted mean direction in (η, φ)
    space, projected back through the unit circle to handle the φ wrap-around.
    Log energy quantities are rescaled relative to the jet-level totals so
    that the features are invariant to the absolute jet energy scale.
    Parameters
    ----------
    particles : ndarray of shape (N, 4), columns [px, py, pz, E]
    Returns
    -------
    ndarray of shape (N, 6), columns [Δη, Δφ, log pT, log E, log pT_rel, log E_rel]
    """
    pT, eta, phi, E = _extract_kinematics(particles)
    pT_sum = pT.sum() + 1e-9
    E_sum = E.sum() + 1e-9
    eta_jet = np.sum(pT * eta) / pT_sum
    phi_jet = np.arctan2(
        np.sum(pT * np.sin(phi)) / pT_sum,
        np.sum(pT * np.cos(phi)) / pT_sum,
    )
    d_eta = eta - eta_jet
    d_phi = _wrap_phi(phi - phi_jet)

    # Ensure arguments to log are strictly positive to avoid NaNs
    log_pT = np.log(np.maximum(pT, 1e-6))
    log_E = np.log(np.maximum(E, 1e-6))
    log_pT_rel = np.log(np.maximum(pT / pT_sum, 1e-6))
    log_E_rel = np.log(np.maximum(E / E_sum, 1e-6))

    return np.stack([d_eta, d_phi, log_pT, log_E, log_pT_rel, log_E_rel],
                    axis=1).astype(np.float32)

# ---------------------------------------------------------------------------
# Dataset class
# ---------------------------------------------------------------------------

class QGJetDataset(Dataset):
    """
    Converts raw jet data from the Zenodo 3164691 HDF5 files into a list of
    PyTorch Geometric Data objects, each representing a single jet as a
    k-NN graph in angular (Δη, Δφ) space.
    Parameters
    ----------
    filepath : str
        Path to the HDF5 file (train.h5 or test.h5).
    split : str
        'train', 'val', or 'test'. Train and val are carved from the same
        file; test should be called on the dedicated test file.
    val_fraction : float
        Fraction of the file's jets reserved for validation.
    max_jets : int, optional
        Cap on the number of jets loaded. Useful for rapid prototyping.
    k : int
        Number of nearest neighbours per node (default 16).
    seed : int
        Random seed for the train/val split.
    """

    N_NODE_FEATURES = 6

    def __init__(self, filepath, split='train', val_fraction=0.15,
                 max_jets=None, k=16, seed=42):
        super().__init__()
        self.k = k
        self.graphs = self._load(filepath, split, val_fraction, max_jets, seed)

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _load(self, filepath, split, val_fraction, max_jets, seed):
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"Dataset file not found: {filepath}")
        if not h5py.is_hdf5(filepath):
            raise OSError(f"File {filepath} is not a valid HDF5 file. It might be corrupted or incomplete.")

        with h5py.File(filepath, 'r') as f:
            # Accept both 'X'/'y' and other common naming conventions
            x_key = 'X' if 'X' in f else next(k for k in f if k.lower().startswith('x'))
            y_key = 'y' if 'y' in f else next(k for k in f if k.lower().startswith('y'))
            X = f[x_key][:]
            y = f[y_key][:].astype(np.int64)

        if max_jets is not None:
            X, y = X[:max_jets], y[:max_jets]

        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(y))
        n_val = int(len(y) * val_fraction)

        if split == 'val':
            idx = idx[:n_val]
        elif split == 'train':
            idx = idx[n_val:]
        # 'test': use all indices in the dedicated test file

        return [self._to_graph(X[i], int(y[i])) for i in idx]

    def _to_graph(self, raw: np.ndarray, label: int) -> Data:
        """
        Convert a single jet's zero-padded particle array to a PyG Data object.
        Only particles with pT > 1e-6 GeV are retained. For the rare edge case
        where fewer than two valid particles remain, a minimal graph with no
        edges is returned so the DataLoader batch collation remains consistent.
        """
        pT, _, _, _ = _extract_kinematics(raw)
        valid = pT > 1e-6
        particles = raw[valid]

        y_tensor = torch.tensor([label], dtype=torch.long)

        if len(particles) < 2:
            x = torch.zeros((max(1, len(particles)), self.N_NODE_FEATURES),
                            dtype=torch.float)
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            return Data(x=x, edge_index=edge_index, y=y_tensor)

        # Construct six-dimensional node feature matrix
        node_feats = _node_features(particles)

        # k-NN graph in (Δη, Δφ) space — physically motivated spatial graph
        coords = node_feats[:, :2]
        k = min(self.k, len(particles) - 1)

        nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='ball_tree').fit(coords)
        _, indices = nbrs.kneighbors(coords)

        n = len(particles)
        src = np.repeat(np.arange(n), k)
        dst = indices[:, 1:].flatten()      # column 0 is the self-neighbour

        edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)
        x = torch.tensor(node_feats, dtype=torch.float)

        return Data(x=x, edge_index=edge_index, y=y_tensor)

    # ------------------------------------------------------------------
    # Dataset protocol
    # ------------------------------------------------------------------

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        return self.graphs[idx]


Dynamic Graph CNN (DGCNN) for Quark/Gluon Jet Classification
==============================================================
Architecture based on:
    Wang et al. (2019) "Dynamic Graph CNN for Learning on Point Clouds"
    https://arxiv.org/abs/1801.07829

Core concept: EdgeConv with dynamic graph recomputation
--------------------------------------------------------
Standard graph convolutions operate on a fixed graph that is constructed once
from the raw input coordinates. DGCNN departs from this by recomputing the
k-nearest-neighbour graph after each EdgeConv layer in the current feature
space. The motivation for this in jet physics is the following:

  - Layer 0: the graph is built in angular (Δη, Δφ) space, capturing
    geometrically adjacent particles.
  - Layer 1+: the graph is rebuilt in the learned feature space. Particles
    that are separated angularly may share learned representations (e.g.
    two hard-fragmentation products of the same splitting) and will be
    connected; conversely, particles that are close in angle but otherwise
    dissimilar may no longer be neighbours.

This progressive re-clustering allows the model to discover long-range
correlations in jet substructure that a fixed angular graph would miss.

EdgeConv operation
------------------
For each node i with k-nearest neighbours {j}: the edge feature between
i and j is the concatenation [h_i, h_j − h_i].  The term h_j − h_i
explicitly encodes the relative displacement in feature space, making each
node aware of its local context while remaining invariant to global
translations of the feature vectors.  A shared MLP is applied to each edge
feature and max-pooling aggregates across the neighbourhood:

    h_i' = max_{j ∈ N(i)} MLP([h_i, h_j − h_i])

Multi-scale readout
-------------------
The intermediate features from all three EdgeConv layers are concatenated
before global pooling.  This allows the classifier head to simultaneously
draw on fine-grained local structure (early layers, small k neighbourhoods)
and coarser global correlations (later layers in learned feature space).
Global mean and max pooling are both applied and concatenated, doubling the
representational capacity of the graph-level embedding.

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import EdgeConv, global_mean_pool, global_max_pool


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

class SharedMLP(nn.Sequential):
    """
    A multi-layer perceptron with BatchNorm and ReLU activations applied
    uniformly to all edges (hence "shared" across edge instances).

    Parameters
    ----------
    channels : list of int
        Sequence of layer widths, e.g. [128, 64, 64].
    """
    def __init__(self, channels):
        layers = []
        for in_c, out_c in zip(channels[:-1], channels[1:]):
            layers += [
                nn.Linear(in_c, out_c, bias=False),
                nn.BatchNorm1d(out_c),
                nn.LeakyReLU(negative_slope=0.2, inplace=True),
            ]
        super().__init__(*layers)


def _knn_graph(x: torch.Tensor, k: int, batch: torch.Tensor) -> torch.Tensor:
    """
    Compute a directed k-nearest-neighbour graph in the current feature space
    using pairwise Euclidean distances.

    This pure-PyTorch implementation processes each graph in the batch
    independently, ensuring that neighbours are never drawn from a different
    jet.  For the typical jet size (20–80 particles) the O(N²) distance matrix
    is inexpensive to compute on modern hardware.

    Parameters
    ----------
    x     : (N_total, F)  — concatenated node features for the whole batch
    k     : int           — number of nearest neighbours (excluding self)
    batch : (N_total,)    — batch assignment vector

    Returns
    -------
    edge_index : (2, E)  — COO edge index (source, destination)
    """
    device = x.device
    src_list, dst_list = [], []

    for bid in batch.unique():
        mask = batch == bid
        x_b = x[mask]
        n = x_b.size(0)
        k_b = min(k, n - 1)
        if k_b == 0:
            continue

        # Squared pairwise distances via the identity ‖a − b‖² = ‖a‖² + ‖b‖² − 2⟨a,b⟩
        dist = torch.cdist(x_b, x_b)
        _, nn_idx = dist.topk(k_b + 1, dim=1, largest=False)
        nn_idx = nn_idx[:, 1:]          # drop the self-loop (distance = 0)

        global_idx = mask.nonzero(as_tuple=True)[0]
        src = global_idx.repeat_interleave(k_b)
        dst = global_idx[nn_idx.reshape(-1)]
        src_list.append(src)
        dst_list.append(dst)

    if not src_list:
        return torch.zeros((2, 0), dtype=torch.long, device=device)

    return torch.stack([torch.cat(src_list), torch.cat(dst_list)])


# ---------------------------------------------------------------------------
# DGCNN building block
# ---------------------------------------------------------------------------

class DynamicEdgeConv(nn.Module):
    """
    EdgeConv layer with in-forward graph recomputation.

    The k-NN graph is rebuilt at call time from the current node embeddings,
    making it "dynamic" across layers.  This is the distinguishing feature of
    DGCNN compared to architectures with a fixed input graph.
    """

    def __init__(self, in_channels: int, out_channels: int, k: int = 16):
        super().__init__()
        self.k = k
        # EdgeConv receives the concatenated [h_i || h_j − h_i] edge feature
        self.conv = EdgeConv(
            nn=SharedMLP([2 * in_channels, out_channels, out_channels]),
            aggr='max',
        )

    def forward(self, x: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        edge_index = _knn_graph(x, self.k, batch)
        return self.conv(x, edge_index)


# ---------------------------------------------------------------------------
# Full DGCNN model
# ---------------------------------------------------------------------------

class DGCNN(nn.Module):
    """
    Three-layer Dynamic Graph CNN with multi-scale feature concatenation,
    dual global pooling, and a two-layer classifier head.

    Layer dimensions
    ----------------
        Input projection  :  in_channels → 64
        DynamicEdgeConv 1 :  64  → 64
        DynamicEdgeConv 2 :  64  → 128
        DynamicEdgeConv 3 :  128 → 256
        Global pool       :  concat(mean, max) over [64 + 128 + 256] = 896
        FC 1              :  896 → 512
        FC 2              :  512 → 256
        Output            :  256 → 2  (logits)

    Parameters
    ----------
    in_channels : int
        Dimensionality of input node features (6 for this dataset).
    k : int
        Number of nearest neighbours used in each EdgeConv layer.
    dropout : float
        Dropout probability applied in the classifier head.
    """

    def __init__(self, in_channels: int = 6, k: int = 16, dropout: float = 0.5):
        super().__init__()
        self.k = k

        # Initial linear projection to normalise the raw input feature scale
        self.input_proj = nn.Sequential(
            nn.Linear(in_channels, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
        )

        self.ec1 = DynamicEdgeConv(64,  64,  k)
        self.ec2 = DynamicEdgeConv(64,  128, k)
        self.ec3 = DynamicEdgeConv(128, 256, k)

        # (64 + 128 + 256) × 2 channels after mean and max global pooling
        pool_dim = (64 + 128 + 256) * 2
        self.classifier = nn.Sequential(
            nn.Linear(pool_dim, 512, bias=False),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 2),
        )

    def forward(self, data):
        x, batch = data.x, data.batch

        x = self.input_proj(x)

        h1 = self.ec1(x, batch)
        h2 = self.ec2(h1, batch)
        h3 = self.ec3(h2, batch)

        # Concatenate feature maps from all three scales before readout.
        # This preserves both fine-grained local structure (h1) and abstract
        # long-range correlations (h3) in the final graph representation.
        h = torch.cat([h1, h2, h3], dim=1)

        # Dual pooling: mean captures the average jet activity; max preserves
        # the dominant substructure signal regardless of multiplicity.
        out = torch.cat([global_mean_pool(h, batch),
                         global_max_pool(h, batch)], dim=1)

        return self.classifier(out)


Graph Attention Network (GAT) for Quark/Gluon Jet Classification
=================================================================
Architecture based on:
    Veličković et al. (2018) "Graph Attention Networks"
    https://arxiv.org/abs/1710.10903

Design philosophy: attention over a fixed angular graph
-------------------------------------------------------
In contrast to the DGCNN, which dynamically rewires the graph in learned
feature space, this model fixes the graph at the structure built from the
physical (Δη, Δφ) angular space and instead learns to weight the contribution
of each neighbour through a trainable attention mechanism.

This is physically motivated for jet classification:

  - The angular graph encodes well-understood QCD geometry: collinear
    splittings create clusters of nearby particles, while soft wide-angle
    radiation is structurally different from hard collinear fragments.
    Preserving this geometry throughout every layer ensures that the model
    operates on physically interpretable neighbourhoods.

  - Not all particles within an angular neighbourhood are equally relevant to
    the substructure signal. Hard collinear daughters of a splitting carry
    most of the discriminating information, while soft underlying-event
    particles add noise. Attention coefficients let the model learn this
    weighting directly from data, without requiring hand-crafted importance
    scores.

  - Multi-head attention spreads representational capacity across independent
    attention subspaces, capturing different aspects of inter-particle
    relationships simultaneously. One head may learn to focus on hard
    splittings, another on soft radiation patterns, etc. Averaging over heads
    at the output layer reduces variance without sacrificing expressiveness.

Attention mechanism
-------------------
For each directed edge (i → j) with node embeddings h_i and h_j, the
attention coefficient is computed as:

    e_ij  = LeakyReLU( a^T [W h_i ∥ W h_j] )
    α_ij  = softmax_j( e_ij )

The updated representation for node i is then:

    h_i'  = σ( Σ_{j ∈ N(i)} α_ij W h_j )

With K parallel attention heads, the outputs are either concatenated
(intermediate layers) or averaged (final layer).

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool


# ---------------------------------------------------------------------------
# GAT-based jet classifier
# ---------------------------------------------------------------------------

class GATJetClassifier(nn.Module):
    """
    Four-layer Graph Attention Network with residual connections, multi-head
    attention, and a two-layer classifier head.

    Layer dimensions (with heads=8, concat=True)
    ---------------------------------------------
        Input projection    :  in_channels → 64           (Linear + BN + ELU)
        GATConv 1           :  64  → 64 × 8  = 512        (8 heads, concat)
        GATConv 2           :  512 → 64 × 8  = 512        (8 heads, concat)
        GATConv 3           :  512 → 64 × 8  = 512        (8 heads, concat)
        GATConv 4           :  512 → 256                  (1 head, mean)
        Global pool         :  concat(mean, max) → 512
        FC 1                :  512 → 256
        FC 2                :  256 → 64
        Output              :  64  → 2  (logits)

    The ELU activation is used throughout instead of ReLU because it provides
    a smooth gradient for negative inputs, which can help convergence in deep
    networks with BatchNorm where pre-activation values may be negative.

    Parameters
    ----------
    in_channels : int
        Input node feature dimensionality (6 for this dataset).
    heads : int
        Number of parallel attention heads in GATConv layers 1–3.
    dropout : float
        Dropout probability applied to attention coefficients and classifier.
    """

    def __init__(self, in_channels: int = 6, heads: int = 8, dropout: float = 0.4):
        super().__init__()
        self.dropout = dropout
        hidden = 64

        # Project raw node features to a uniform embedding before attention
        self.input_proj = nn.Sequential(
            nn.Linear(in_channels, hidden, bias=False),
            nn.BatchNorm1d(hidden),
            nn.ELU(inplace=True),
        )

        # Intermediate layers concatenate across heads, expanding the width
        self.gat1 = GATConv(hidden,          hidden, heads=heads, dropout=dropout, concat=True)
        self.gat2 = GATConv(hidden * heads,  hidden, heads=heads, dropout=dropout, concat=True)
        self.gat3 = GATConv(hidden * heads,  hidden, heads=heads, dropout=dropout, concat=True)

        # Final attention layer uses a single head and averages, producing a
        # fixed-width representation independent of the head count
        self.gat4 = GATConv(hidden * heads, 256, heads=1, dropout=dropout, concat=False)

        self.bn1 = nn.BatchNorm1d(hidden * heads)
        self.bn2 = nn.BatchNorm1d(hidden * heads)
        self.bn3 = nn.BatchNorm1d(hidden * heads)
        self.bn4 = nn.BatchNorm1d(256)

        # Classifier receives the concatenated global mean and max pooled features
        self.classifier = nn.Sequential(
            nn.Linear(256 * 2, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ELU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.ELU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.input_proj(x)

        # Attention layers with residual connections where dimensionality permits.
        # The residual from the input projection is reused across layers 1–3
        # because they share the same hidden × heads width; this improves
        # gradient flow and mitigates over-smoothing in deep GAT stacks.
        x1 = F.elu(self.bn1(self.gat1(x, edge_index)))
        x2 = F.elu(self.bn2(self.gat2(x1, edge_index))) + x1
        x3 = F.elu(self.bn3(self.gat3(x2, edge_index))) + x2
        x4 = F.elu(self.bn4(self.gat4(x3, edge_index)))

        # Graph-level readout: global mean captures the average particle
        # representation while global max preserves the most prominent signal
        out = torch.cat([global_mean_pool(x4, batch),
                         global_max_pool(x4, batch)], dim=1)

        return self.classifier(out)


Training and Evaluation Script
================================
Trains and evaluates the DGCNN and GAT architectures on the quark/gluon jet
classification task and produces a side-by-side performance comparison.

In [13]:
import numpy as np
import h5py
import os
from sklearn.model_selection import train_test_split

# Define the path to the provided .npz file
npz_file_path = '/content/QG_jets_10.npz'

# Create 'data' directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Load data from the .npz file (corrected to use np.load)
print(f"Loading data from {npz_file_path}...")
try:
    loaded_data = np.load(npz_file_path, allow_pickle=True)
    X = loaded_data['X']  # Access dataset 'X'
    y = loaded_data['y']  # Access dataset 'y'
    print(f"Loaded {len(X)} jets. X shape: {X.shape}, y shape: {y.shape}")

    # Split data into training and testing sets
    # Assuming a typical 80/20 split for train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Train set: {len(X_train)} jets, Test set: {len(X_test)} jets")

    # Define output HDF5 file paths
    train_h5_path = 'data/train.h5'
    test_h5_path = 'data/test.h5'

    # Save training data to HDF5
    with h5py.File(train_h5_path, 'w') as f:
        f.create_dataset('X', data=X_train)
        f.create_dataset('y', data=y_train)
    print(f"Training data saved to {train_h5_path}")

    # Save test data to HDF5
    with h5py.File(test_h5_path, 'w') as f:
        f.create_dataset('X', data=X_test)
        f.create_dataset('y', data=y_test)
    print(f"Test data saved to {test_h5_path}")

except Exception as e:
    print(f"Error loading {npz_file_path}: {e}")
    print("Please ensure the file is a valid .npz format, or check its contents.")


Loading data from /content/QG_jets_10.npz...
Loaded 100000 jets. X shape: (100000, 138, 4), y shape: (100000,)
Train set: 80000 jets, Test set: 20000 jets
Training data saved to data/train.h5
Test data saved to data/test.h5


In [14]:
import os
import argparse
import time
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Model factory
# ---------------------------------------------------------------------------

def build_model(name: str, in_channels: int, device: torch.device) -> nn.Module:
    """Instantiate a model by name and move it to the target device."""
    if name == 'dgcnn':
        return DGCNN(in_channels=in_channels, k=16, dropout=0.5).to(device)
    elif name == 'gat':
        return GATJetClassifier(in_channels=in_channels, heads=8, dropout=0.4).to(device)
    else:
        raise ValueError(f"Unknown model '{name}'. Choose from: dgcnn, gat")

# ---------------------------------------------------------------------------
# Training utilities
# ---------------------------------------------------------------------------

def train_one_epoch(model, loader, optimiser, criterion, device):
    """Run a single training epoch and return mean loss."""
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimiser.zero_grad()
        logits = model(batch)
        # y is stored as (N, 1); squeeze to (N,) for cross-entropy
        loss = criterion(logits, batch.y.squeeze())
        loss.backward()
        # Gradient clipping avoids occasional divergence from sparse attention
        # gradients in the GAT when learning rates are not yet warmed up
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    """
    Evaluate a model on a DataLoader and return a results dictionary.

    Returns
    -------
    dict with keys: loss, accuracy, auc, y_true, y_score
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    all_true, all_score = [], []

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        y = batch.y.squeeze()
        loss = criterion(logits, y)
        total_loss += loss.item() * batch.num_graphs

        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_true.extend(y.cpu().numpy())
        all_score.extend(probs)

    y_true = np.array(all_true)
    y_score = np.array(all_score)
    y_pred = (y_score >= 0.5).astype(int)

    return {
        'loss': total_loss / len(loader.dataset),
        'accuracy': accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_pred),
        'y_true': y_true,
        'y_score': y_score,
    }

# ---------------------------------------------------------------------------
# Full training loop
# ---------------------------------------------------------------------------

def train_model(model, name, train_loader, val_loader, args, device):
    """
    Train a model with cosine-annealing learning rate schedule and early
    stopping on validation AUC.

    Returns the best validation AUC achieved and the model state at that point.
    """
    criterion = nn.CrossEntropyLoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=args.lr,
                                 weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=args.epochs, eta_min=args.lr * 1e-2
    )

    best_auc = 0.0
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

    print(f"\n{'=' * 60}")
    print(f"  Training {name.upper()}")
    print(f"{'=' * 60}")
    print(f"  Parameters : {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Epochs     : {args.epochs}")
    print(f"  Batch size : {args.batch_size}")
    print(f"  Device     : {device}")
    print()

    t0 = time.time()

    for epoch in range(1, args.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimiser,
                                     criterion, device)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_auc'].append(val_metrics['auc'])

        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 5 == 0 or epoch == 1:
            elapsed = time.time() - t0
            print(f"  Epoch {epoch:3d}/{args.epochs}  |  "
                  f"train loss: {train_loss:.4f}  |  "
                  f"val loss: {val_metrics['loss']:.4f}  |  "
                  f"val AUC: {val_metrics['auc']:.4f}  |  "
                  f"{elapsed:.0f}s elapsed")

        if patience_counter >= args.patience:
            print(f"\n  Early stopping at epoch {epoch} "
                  f"(no improvement for {args.patience} epochs).")
            break

    print(f"\n  Best validation AUC: {best_auc:.4f}")

    # Restore best weights
    model.load_state_dict(best_state)

    # Save checkpoint
    ckpt_path = os.path.join(args.models_dir, f"{name}_best.pt")
    torch.save({'model_state': best_state, 'args': vars(args)}, ckpt_path)
    print(f"  Checkpoint saved -> {ckpt_path}")

    return best_auc, history

# ---------------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------------

def plot_training_history(histories: dict, output_path: str = "training_history.png"):
    """Plot training loss and validation AUC curves for all models."""
    n_models = len(histories)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    colours = {'dgcnn': 'steelblue', 'gat': 'tomato'}
    for name, hist in histories.items():
        colour = colours.get(name, 'grey')
        epochs = range(1, len(hist['train_loss']) + 1)
        axes[0].plot(epochs, hist['train_loss'], label=f"{name.upper()} train",
                     color=colour, linestyle='--', alpha=0.7)
        axes[0].plot(epochs, hist['val_loss'], label=f"{name.upper()} val",
                     color=colour)
        axes[1].plot(epochs, hist['val_auc'], label=name.upper(), color=colour)

    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].set_title("Training and Validation Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("ROC AUC")
    axes[1].set_title("Validation ROC AUC")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\nTraining history plot saved -> {output_path}")

def plot_roc_curves(results: dict, output_path: str = "roc_comparison.png"):
    """Plot ROC curves for all models on the test set."""
    fig, ax = plt.subplots(figsize=(7, 6))

    colours = {'dgcnn': 'steelblue', 'gat': 'tomato'}

    for name, metrics in results.items():
        fpr, tpr, _ = roc_curve(metrics['y_true'], metrics['y_score'])
        auc = metrics['auc']
        colour = colours.get(name, 'grey')
        ax.plot(fpr, tpr, label=f"{name.upper()}  (AUC = {auc:.4f})",
                color=colour, linewidth=2)

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
    ax.set_xlabel("False Positive Rate (gluon misidentification)")
    ax.set_ylabel("True Positive Rate (quark efficiency)")
    ax.set_title("ROC Curve — Quark vs. Gluon Jet Classification")
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"ROC comparison plot saved  -> {output_path}")

def print_results_table(results: dict):
    """Print a formatted summary table of test-set metrics."""
    print(f"\n{'=' * 55}")
    print(f"  Test Set Results")
    print(f"{'=' * 55}")
    print(f"  {'Model':<10}  {'Accuracy':>10}  {'ROC AUC':>10}")
    print(f"  {'-' * 10}  {'-' * 10}  {'-' * 10}")
    for name, metrics in results.items():
        print(f"  {name.upper():<10}  {metrics['accuracy']:>10.4f}  "
              f"{metrics['auc']:>10.4f}")
    print(f"{'=' * 55}\n")

def parse_args(argv=None):
    parser = argparse.ArgumentParser(
        description="Train GNN models for quark/gluon jet classification"
    )
    parser.add_argument('--model', choices=['dgcnn', 'gat'], default='dgcnn',
                        help="Model to train (ignored when --compare is set)")
    parser.add_argument('--compare', action='store_true',
                        help="Train both models and produce comparison output")
    parser.add_argument('--train', default='data/train.h5',
                        help="Path to training HDF5 file")
    parser.add_argument('--test', default='data/test.h5',
                        help="Path to test HDF5 file")
    parser.add_argument('--epochs', type=int, default=20)
    parser.add_argument('--batch-size', type=int, default=128)
    parser.add_argument('--lr', type=float, default=1e-3)
    parser.add_argument('--patience', type=int, default=15,
                        help="Early stopping patience (epochs)")
    parser.add_argument('--k', type=int, default=16,
                        help="k-NN connectivity for graph construction")
    parser.add_argument('--max-jets', type=int, default=None,
                        help="Limit dataset size for debugging")
    parser.add_argument('--val-fraction', type=float, default=0.15)
    parser.add_argument('--num-workers', type=int, default=0,
                        help="DataLoader worker processes")
    parser.add_argument('--models-dir', type=str, default='models',
                        help="Directory where model checkpoints are saved")
    parser.add_argument('--figures-dir', type=str, default='figures_task2_new',
                        help="Directory where figure outputs are saved")
    return parser.parse_args(argv)

def main():
    args = parse_args([]) # Pass an empty list to avoid parsing kernel arguments
    os.makedirs(args.models_dir, exist_ok=True)
    os.makedirs(args.figures_dir, exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")

    # ------------------------------------------------------------------
    # Load datasets
    # ------------------------------------------------------------------
    print("\nBuilding training and validation graphs…")
    t0 = time.time()
    train_ds = QGJetDataset(args.train, split='train',
                            val_fraction=args.val_fraction,
                            max_jets=args.max_jets, k=args.k)
    val_ds = QGJetDataset(args.train, split='val',
                          val_fraction=args.val_fraction,
                          max_jets=args.max_jets, k=args.k)
    print(f"  Train: {len(train_ds):,} jets  |  Val: {len(val_ds):,} jets  "
          f"({time.time() - t0:.1f}s)")

    print("Building test graphs…")
    t0 = time.time()
    test_ds = QGJetDataset(args.test, split='test',
                           max_jets=args.max_jets, k=args.k)
    print(f"  Test : {len(test_ds):,} jets  ({time.time() - t0:.1f}s)")

    # Node feature dimensionality is inferred from the first graph
    in_channels = train_ds[0].x.size(1)

    kw = dict(batch_size=args.batch_size, num_workers=args.num_workers,
              pin_memory=(device.type == 'cuda'))
    train_loader = DataLoader(train_ds, shuffle=True, **kw)
    val_loader = DataLoader(val_ds, shuffle=False, **kw)
    test_loader = DataLoader(test_ds, shuffle=False, **kw)

    # ------------------------------------------------------------------
    # Determine which models to run
    # ------------------------------------------------------------------
    model_names = ['dgcnn', 'gat'] if args.compare else [args.model]

    histories = {}
    test_results = {}

    for name in model_names:
        model = build_model(name, in_channels, device)
        _, history = train_model(model, name, train_loader, val_loader, args, device)
        histories[name] = history

        print(f"\nEvaluating {name.upper()} on test set…")
        metrics = evaluate(model, test_loader, device)
        test_results[name] = metrics
        print(f"  Accuracy : {metrics['accuracy']:.4f}")
        print(f"  ROC AUC  : {metrics['auc']:.4f}")

    # ------------------------------------------------------------------
    # Summary and plots
    # ------------------------------------------------------------------
    if len(model_names) > 1:
        print_results_table(test_results)
        plot_roc_curves(test_results, os.path.join(args.figures_dir, "roc_comparison.png"))
    else:
        name = model_names[0]
        fpr, tpr, _ = roc_curve(test_results[name]['y_true'],
                                 test_results[name]['y_score'])
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, color='steelblue',
                label=f"AUC = {test_results[name]['auc']:.4f}")
        ax.plot([0, 1], [0, 1], 'k--')
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC Curve — {name.upper()}")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        roc_path = os.path.join(args.figures_dir, f"roc_{name}.png")
        fig.savefig(roc_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"ROC curve saved -> {roc_path}")

    plot_training_history(histories, os.path.join(args.figures_dir, "training_history.png"))

if __name__ == "__main__":
    main()


Device: cuda

Building training and validation graphs…
  Train: 68,000 jets  |  Val: 12,000 jets  (122.1s)
Building test graphs…
  Test : 20,000 jets  (29.2s)

  Training DGCNN
  Parameters : 770,306
  Epochs     : 20
  Batch size : 128
  Device     : cuda



/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


  Epoch   1/20  |  train loss: 0.4855  |  val loss: 0.4742  |  val AUC: 0.7790  |  150s elapsed


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


  Epoch   5/20  |  train loss: 0.4515  |  val loss: 0.5396  |  val AUC: 0.7437  |  752s elapsed


...

  Epoch  10/20  |  train loss: 0.4327  |  val loss: 0.4316  |  val AUC: 0.8064  |  1508s elapsed


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

  Epoch  15/20  |  train loss: 0.4141  |  val loss: 0.4340  |  val AUC: 0.8072  |  2267s elapsed


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

  Epoch  20/20  |  train loss: 0.3957  |  val loss: 0.4273  |  val AUC: 0.8105  |  3025s elapsed

  Best validation AUC: 0.8124
  Checkpoint saved -> models/dgcnn_best.pt

Evaluating DGCNN on test set…
  Accuracy : 0.8115
  ROC AUC  : 0.8115
ROC curve saved -> figures_task2_new/roc_dgcnn.png

Training history plot saved -> figures_task2_new/training_history.png
